### 졸음감지 전이학습 이미지 데이터 전처리 

In [42]:
import os
import cv2
import copy 
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import torch 
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [75]:
# check length of datasets 
dset1_close = os.listdir('./dataset_1/train/Closed_Eyes')
dset1_open = os.listdir('./dataset_1/train/Open_Eyes')
dset2_tr_close = os.listdir('./dataset_2/TrainingSet/TrainingSet/Closed')
dset2_tr_open = os.listdir('./dataset_2/TrainingSet/TrainingSet/Opened')
dset2_ts_close = os.listdir('./dataset_2/TestSet/TestSet/Closed')
dset2_ts_open = os.listdir('./dataset_2/TestSet/TestSet/Opened')
dset3_tr_close = os.listdir('./dataset_3/dataset_new/train/Close')
dset3_tr_open = os.listdir('./dataset_3/dataset_new/train/Open')
dset3_ts_close = os.listdir('./dataset_3/dataset_new/test/Close')
dset3_ts_open = os.listdir('./dataset_3/dataset_new/test/Open')

print('dataset1 closed:', len(dset1_close))
print('dataset1 open:', len(dset1_open))
print('dataset2 train closed:',len(dset2_tr_close))
print('dataset2 train open:', len(dset2_tr_open))
print('dataset2 test closed:', len(dset2_ts_close))
print('dataset2 test open:', len(dset2_ts_open))
print('dataset3 train close:', len(dset3_tr_close))
print('dataset3 train open:', len(dset3_tr_open))
print('dataset3 test close:', len(dset3_ts_close))
print('dataset3 test open:', len(dset3_ts_open))

dataset1 closed: 2000
dataset1 open: 2000
dataset2 train closed: 880
dataset2 train open: 824
dataset2 test closed: 2190
dataset2 test open: 2042
dataset3 train close: 617
dataset3 train open: 617
dataset3 test close: 109
dataset3 test open: 109


In [96]:
train_dir = './train/'
test_dir = './test/'
dset1_dir = './dataset_1/train/'

In [119]:
def list_image_files(data_dir, sub_dir, tr_ts=None):
    image_format = ['jpeg', 'jpg', 'png']
    image_files = []
    image_dir = os.path.join(data_dir, sub_dir)
    for i, file_path in enumerate(os.listdir(image_dir)):
        if tr_ts == None:
            if file_path.split('.')[-1] in image_format:
                image_files.append(os.path.join(sub_dir, file_path))
        elif tr_ts == 'test':
            if i%4 == 0:
                if file_path.split('.')[-1] in image_format:
                    image_files.append(os.path.join(sub_dir, file_path))        
        elif tr_ts == 'train':
            if i%4 != 0:
                if file_path.split('.')[-1] in image_format:
                    image_files.append(os.path.join(sub_dir, file_path))
    return image_files 

In [ ]:
def get_RGB_image(file_path_list):
    image_file = os.path.join(data_dir, file_name)
    image = cv2.imread(image_file)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image

In [171]:
class Eye_dataset(Dataset):
    def __init__(self, data_dir, dset1_dir, tr_ts, transform=None): 
        self.data_dir = data_dir
        self.dset1_dir = dset1_dir 
        self.classes = ['close', 'open']
        self.close = list_image_files(dset1_dir, 'close', tr_ts) + list_image_files(data_dir, 'close') 
        self.open = list_image_files(dset1_dir, 'open', tr_ts) + list_image_files(data_dir, 'open')  
        self.files_path = self.close + self.open 
        self.transform = transform 
        
    def __len__(self):
        return len(self.files_path)

    def __getitem__(self, index):
        if self.files_path[index].split('.')[-1] == 'png':
           image_file = os.path.join(self.dset1_dir, self.files_path[index])
        else:
           image_file = os.path.join(self.data_dir, self.files_path[index])
        image = cv2.imread(image_file)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        target = self.classes.index(self.files_path[index].split(os.sep)[-2])
        return {'image':image, 'target':target}